# 14 - Sensibilidad al umbral de viabilidad

Habíamos establecido un umbral del 50% sobre la viabilidad para establecer los conjuntos de dimensiones según el origen de captura de los datos.

Recordemos que la viabilidad se ha calculado de manera subjetiva en base a criterios del autor.

Vamos a realizar un análisis de sensibilidad con diferentes valores para el umbral: `config.THRESHOLDS` (40, 50, 60, 75 y 90).

Los resultados se generan ejecutando `make sensitivity` (`reports/results/sensitivity.csv`), con los hiperparámetros de `best_params.json` cuando se han calculado para el subconjunto.

Cuando un conjunto no tiene hiperparámetros propios (porque sus dimensiones no coinciden con ningún conjunto ajustado antes) se usan los de completo. La columna used_fallback marca esas filas.

El conjunto `completo` usa las 7 dimensiones, así que no depende del umbral y no lo recalculamos.

## Carga de datos

In [ ]:
from IPython.display import Markdown, display
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import (
    FEATURE_SETS,
    MATRIX_MODELS,
    PROFILES,
    REPORTS_DIR,
    THRESHOLDS,
)
from urban_accessibility_ml.modeling.evaluate import summarize_folds

SENSITIVITY_FILE = REPORTS_DIR / "results" / "sensitivity.csv"
TUNED_FILE = REPORTS_DIR / "results" / "experiment-matrix-tuned.csv"
STRATEGIES = ["within_city", "mixed", "cross_city"]
CLEARLY_INFERIOR_RATIO = 0.8
SUBSETS = [feature_set for feature_set in FEATURE_SETS if feature_set != "completo"]

sensitivity = pd.read_csv(SENSITIVITY_FILE)
tuned = pd.read_csv(TUNED_FILE)
print(len(sensitivity), "filas de sensibilidad,", len(tuned), "filas de la matriz tuned")

## Conjuntos de variables por umbral
Vamos a mostrar a continuación cómo quedan los conjuntos de variables para cada umbral definido.

In [ ]:
summary = summarize_folds(
    sensitivity, ["threshold", "feature_set", "dimensions", "strategy", "profile", "model"]
)

sets = summary[["threshold", "feature_set", "dimensions"]].drop_duplicates()
sets.pivot(index="threshold", columns="feature_set", values="dimensions").reindex(
    columns=SUBSETS
).fillna("(vacío)")

Observamos que el conjunto cartográfico aéreo está vacío a partir del 75% de umbral de viabilidad.

En el caso de 50% y 60%, el conjunto solo contiene cruces y mobiliario.
Y con un 40%: anchura, pavimento, cruces y mobiliario.

Mientras que el de calle tiene la anchura en todos los subconjuntos, incluso en el del 90%.
Después, el mobiliario se incluye con el 60% y el 75%.
Con el 50% se añaden cruces y señalización, mientras que con el 40% se incluye pendiente.

Observamos también que en el conjunto combinado, únicamente en los casos del 40% y del 60% se podría realizar el contraste de hipótesis H3.
En el caso del 40%, pavimento solo está en cartográfico aéreo, mientras que pendiente y cruces en calle.
En el caso del 60%, aéreo tiene cruces, y calle anchura.
En el resto de casos, calle tiene más dimensiones, por lo que el combinado es igual a calle (50%, 75% y 90%).

## Resumen por umbral, conjunto, perfil y modelo
Vamos a calcular para los conjuntos, la media y desviación típica entre pliegues por estrategia.

Comparamos cada fila con la F1 macro del mismo modelo, estrategia y perfil con el conjunto completo, que es la misma para todos los umbrales.

In [ ]:
complete = summarize_folds(
    tuned[tuned["feature_set"] == "completo"], ["strategy", "profile", "model"]
)
complete = complete.rename(
    columns={
        "f1_macro_mean": "f1_macro_completo",
        "quadratic_weighted_kappa_mean": "kappa_ponderada_cuadratica_completo",
        "balanced_accuracy_mean": "balanced_accuracy_completo",
    }
)
complete = complete[
    [
        "strategy",
        "profile",
        "model",
        "f1_macro_completo",
        "kappa_ponderada_cuadratica_completo",
        "balanced_accuracy_completo",
    ]
]

n_rows = len(summary)
summary = summary.merge(complete, on=["strategy", "profile", "model"], validate="many_to_one")
assert len(summary) == n_rows, "algún modelo, estrategia o perfil no tiene referencia de completo"
summary["ratio"] = summary["f1_macro_mean"] / summary["f1_macro_completo"]
summary["supera_80"] = summary["ratio"] >= CLEARLY_INFERIOR_RATIO
summary.head()

# F1 macro, kappa ponderada cuadrática y exactitud balanceada

Vamos a calcular en una tabla la F1 macro, kappa ponderada cuadrática y exactitud balanceada, usando la media entre pliegues, de todos los conjuntos.

Incluimos una columna `≥80 %`, para indicar si la F1 macro de cada conjunto es mayor o igual al 80% de la de `completo` con el mismo modelo, estrategia y perfil.

Todas las métricas son las de test de cada pliegue; no se usan para elegir nada, solo para evaluar.

In [ ]:
def table_for(threshold: int, model: str) -> pd.DataFrame:
    data = summary[(summary["threshold"] == threshold) & (summary["model"] == model)]
    columns = {}
    reference = complete[complete["model"] == model].set_index(["strategy", "profile"])
    columns[("completo", "F1 macro")] = reference["f1_macro_completo"]
    columns[("completo", "kappa ponderada cuadrática")] = reference[
        "kappa_ponderada_cuadratica_completo"
    ]
    columns[("completo", "exactitud balanceada")] = reference["balanced_accuracy_completo"]
    for feature_set in SUBSETS:
        rows = data[data["feature_set"] == feature_set].set_index(["strategy", "profile"])
        if rows.empty:
            continue
        columns[(feature_set, "F1 macro")] = rows["f1_macro_mean"]
        columns[(feature_set, "kappa ponderada cuadrática")] = rows[
            "quadratic_weighted_kappa_mean"
        ]
        columns[(feature_set, "exactitud balanceada")] = rows["balanced_accuracy_mean"]
        columns[(feature_set, "≥80 %")] = rows["supera_80"].map({True: "sí", False: "no"})
    index = pd.MultiIndex.from_product([STRATEGIES, PROFILES], names=["strategy", "profile"])
    return pd.DataFrame(columns).reindex(index).round(3)


for threshold in THRESHOLDS:
    for model in MATRIX_MODELS:
        if model == "dummy":
            continue
        display(Markdown(f"### Umbral {threshold} - {model}"))
        display(table_for(threshold, model))

# Conjuntos con menos del 80% de la F1 macro respecto del completo

Vamos a mostrar el número de conjuntos de estrategia y perfil en las que el conjunto queda por debajo del 80 % de la F1 macro de completo con el mismo modelo.

Hay 12 en total por cada conjunto.

In [ ]:
below = summary[summary["model"] != "dummy"].assign(por_debajo=lambda df: ~df["supera_80"])
below.groupby(["threshold", "feature_set", "model"])["por_debajo"].sum().unstack("model")[
    [m for m in MATRIX_MODELS if m != "dummy"]
]

Como podemos ver, la regresión logística es el modelo que más veces queda por debajo del 80 % de `completo`, incluso con los umbrales del 40 % y del 50 %.

El resto de modelos quedan por encima en la mayoría de casos, exceptuando en el umbral del 60%, en el que XGBoost y LightGBM tienen un 25% de los conjuntos de calle y cartográfico/aéreo por debajo.

Con el umbral del 75%, XGBoost y LightGBM también tienen 3 de 12 casos por debajo, y con el 90% los tres modelos de árboles caen en 8 o 9 de 12.

La regresión logística parece tener más dependencia de disponer de todas las dimensiones, mientras que los modelos no lineales se resienten menos.

# Mapa de calor de la F1 macro respecto a `completo` según el umbral

Vamos a representar con un mapa de calor el porcentaje entre la F1 macro de cada conjunto y la de `completo`, con un mapa por modelo.

Cada fila es una estrategia y un perfil, y cada columna un umbral y un conjunto. En negrita están los valores por debajo del 80 % de `completo`.

Cuando `combinado` tiene las mismas dimensiones que `calle`, las dos columnas muestran el mismo valor.

In [ ]:
import matplotlib.pyplot as plt

from urban_accessibility_ml.plotting.plots import save_figure, use_style

use_style()
FIGURES_FOLDER = "0.14-sensibilidad-umbral"
SHORT_NAMES = {"cartografico_aereo": "cart.\naéreo", "calle": "calle", "combinado": "comb."}


def plot_heatmap(model: str) -> None:
    """Dibuja y guarda el mapa de calor de la F1 macro respecto a completo de un modelo."""
    heat = summary[summary["model"] == model].pivot_table(
        index=["strategy", "profile"], columns=["threshold", "feature_set"], values="ratio"
    )
    heat = heat.reindex(
        index=pd.MultiIndex.from_product([STRATEGIES, PROFILES]),
        columns=pd.MultiIndex.from_product([THRESHOLDS, SUBSETS]),
    )
    heat = heat.dropna(axis=1, how="all")  # conjuntos vacíos

    fig, ax = plt.subplots(figsize=(11, 6))
    image = ax.imshow(heat.values, cmap="RdYlGn", vmin=0.5, vmax=1.1, aspect="auto")
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            value = heat.values[i, j]
            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                fontweight="bold" if value < CLEARLY_INFERIOR_RATIO else "normal",
            )
    ax.set_xticks(range(heat.shape[1]), [SHORT_NAMES[fs] for _, fs in heat.columns])
    ax.set_yticks(range(heat.shape[0]), [f"{s} · {p}" for s, p in heat.index])
    for j in range(1, heat.shape[1]):
        if heat.columns[j][0] != heat.columns[j - 1][0]:
            ax.axvline(j - 0.5, color="white", linewidth=3)
    for i in range(len(PROFILES), heat.shape[0], len(PROFILES)):
        ax.axhline(i - 0.5, color="white", linewidth=3)
    for threshold in THRESHOLDS:
        positions = [j for j, (t, _) in enumerate(heat.columns) if t == threshold]
        ax.text(np.mean(positions), -0.9, f"umbral {threshold}", ha="center", fontweight="bold")
    ax.set_title(model, pad=30)
    ax.tick_params(length=0)
    ax.grid(False)
    fig.colorbar(
        image, ax=ax, label="F1 macro respecto a completo (en negrita, por debajo del 80 %)"
    )
    fig.tight_layout()
    save_figure(f"f1_respecto_a_completo_mapa_calor_{model}", FIGURES_FOLDER, fig=fig)
    plt.show()

In [ ]:
plot_heatmap("logistic_regression")

Comprobamos que el umbral que rinde peor es el del 90%, con la mayoría de estrategias y perfiles no llegando al 80% del completo, con una única excepción, cross_city cognitiva.

El umbral del 75% también tiene la mayoría de estrategias y perfiles por debajo del 80% del completo. Otra excepción, esta vez cross_city global.

El umbral del 60% también está por debajo del 80% en la mayoría de casos de mixed y within_city, sin embargo sí que podemos observar algunos valores por encima del 80%, especialmente en cross_city global, o cross_city cognitiva (solo calle está por debajo del 0,80).

Con el umbral del 50% también hay 12 casos por debajo del 80%, 9 de ellos en cartográfico/aéreo.

En general, parece que en el modelo de regresión logística es difícil llegar al 0,80 tras la hiperparametrización.

In [ ]:
plot_heatmap("random_forest")

Hemos comprobado que el umbral que rinde peor es el del 90%, algo que se podría intuir, por el bajo número de dimensiones que contienen los conjuntos de calle o combinado (que es el mismo).

Sin embargo, con el umbral del 75%, solo mixed sensorial queda justo por debajo del 80% del completo (0,797).

El resto de combinaciones está igual o por encima del 80%, exceptuando mixed cognitiva en cartográfico/aéreo, que queda en el 77% del completo con umbral del 50% y del 60%, así como mixed sensorial en calle con el umbral 60%.

In [ ]:
plot_heatmap("xgboost_balanced")

El caso de XGBoost es similar al random forest, con el umbral 90% el que más problemas tiene para llegar al 0,8 del conjunto completo, en within_city y mixed.

Los umbrales 60% y 75% tienen algunos valores por debajo del 0,80 también, específicamente en within_city para sensorial y cognitiva, y en mixed para sensorial.

En el umbral del 60% el conjunto de calle tiene algunos valores por debajo del 0,80, como within_city sensorial y cognitiva, así como mixed sensorial. También cartográfico/aéreo tiene mixed global y cognitiva y cross_city sensorial por debajo.

El resto de combinaciones está por encima del 80%.

In [ ]:
plot_heatmap("lightgbm")

Se repite para LightGBM la misma conclusión para el umbral del 90%, con todos los casos de within_city y mixed por debajo del 80% del F1 macro del conjunto completo. Igual que cross_city para el perfil global.

En cross_city global también se está por debajo del 0,80 en calle con los umbrales del 40% y del 50%, y con el 75% queda justo en el 0,80.

Por último, vemos algunos valores rindiendo por debajo del 0,80, especialmente mixed cognitiva para cartográfico aéreo con umbrales 50% y 60%, o mixed sensorial para calle 60% y 75%.

# Contraste de H3

Vamos a realizar el contraste de la hipótesis 3 ("Combinar las variables cartográficas y aéreas con las de imágenes a pie de calle mejora el rendimiento de los modelos que utilizan únicamente uno de los dos conjuntos") en otro notebook.

Lo que podemos comentar por el momento es que los umbrales en los que se podría contrastar la hipótesis serían: 40% y 60%.